# TCRP Training Experiments

Trains all **dataset × strategy** combinations and collects test-set MSE / MAE for comparison.

| | ETTh1 | ETTm2 | Weather | ExchangeRate | GEFCOM2014 |
|---|:---:|:---:|:---:|:---:|:---:|
| **standard** | ✓ | ✓ | ✓ | ✓ | ✓ |
| **regularized** | ✓ | ✓ | ✓ | ✓ | ✓ |
| **adversarial** | ✓ | ✓ | ✓ | ✓ | ✓ |

**Strategies**
- `standard`   — weighted Pearson alignment only (`lambda4=0`)
- `regularized` — + magnitude alignment (`lambda4=0.1`)
- `adversarial` — regularized + GRL two-path backward (`adversarial=true`)

Results are written to `results/training_experiments.csv` after every run so the notebook survives kernel restarts.

In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path(".").resolve()))

import pandas as pd
import torch

print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"CWD     : {Path('.').resolve()}")

Python  : 3.12.13
PyTorch : 2.11.0+cu128
Device  : cuda
CWD     : /content


In [2]:
from TCRP.tcrp.dataset.datasets import DATASET_META

DATA_ROOT = Path("TCRP/tcrp/data/raw")

available, missing = [], []
for name, meta in DATASET_META.items():
    p = DATA_ROOT / meta["filename"]
    (available if p.exists() else missing).append(name)

print("Available :", available)
print("Missing   :", missing or "none")

Available : ['ETTh1', 'ETTm2', 'Weather', 'ExchangeRate']
Missing   : ['GEFCOM2014']


## 1. Experiment matrix

Edit `DATASETS` and `STRATEGIES` to restrict which combinations are run.

In [3]:
# ── Experiment matrix ─────────────────────────────────────────────────────────
DATASETS = available          # or e.g. ["ETTh1", "ETTm2"]
T, H = 336, 96

STRATEGIES = {
    "standard":    {"adversarial": False, "lambda4": 0.0},
    "regularized": {"adversarial": False, "lambda4": 0.1},
    "adversarial": {"adversarial": True,  "lambda4": 0.1},
}

RESULTS_CSV = Path("results/training_experiments.csv")
RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)

total = len(DATASETS) * len(STRATEGIES)
print(f"{len(DATASETS)} datasets × {len(STRATEGIES)} strategies = {total} experiments")
print(f"Results  → {RESULTS_CSV}")

4 datasets × 3 strategies = 12 experiments
Results  → results/training_experiments.csv


In [4]:
import sys
import os
from pathlib import Path

# Add the INSIDE of the TCRP directory to your path
sys.path.insert(0, os.path.abspath('TCRP'))

## 2. Training helpers

In [5]:
!pip install hydra-core

In [6]:
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf

CONFIGS_DIR = str(Path("TCRP/configs").resolve())

# dataset name → hydra dataset override key
_DS_KEY = {
    "ETTh1":        "etth1",
    "ETTm2":        "ettm2",
    "Weather":      "weather",
    "ExchangeRate": "exchange_rate",
    "GEFCOM2014":   "gefcom2014",
}


def build_cfg(dataset: str, strategy: str) -> OmegaConf:
    """Compose a DictConfig for one (dataset, strategy) pair."""
    s = STRATEGIES[strategy]
    ds_key = _DS_KEY[dataset]
    run_name = f"{dataset}_tcrp_T{T}_H{H}_{strategy}"

    GlobalHydra.instance().clear()
    with initialize_config_dir(config_dir=CONFIGS_DIR, version_base=None):
        cfg = compose(
            config_name="train",
            overrides=[
                f"datasets={ds_key}",
                f"T={T}",
                f"H={H}",
                f"run_name={run_name}",
                f"models.adversarial={'true' if s['adversarial'] else 'false'}",
                f"models.lambda4={s['lambda4']}",
                "checkpoint_dir=checkpoints",
            ],
        )
    return cfg


# quick smoke-test
cfg = build_cfg("ETTh1", "standard")
print("dataset :", cfg.datasets.dataset)
print("lambda4 :", cfg.models.lambda4)
print("advers. :", cfg.models.adversarial)

dataset : ETTh1
lambda4 : 0.0
advers. : False


In [ ]:
from tcrp.pipelines.train import run as train_run


def run_experiment(dataset: str, strategy: str, skip_existing: bool = True) -> dict | None:
    """Train one (dataset, strategy) pair and return its result row.

    Returns None if skipped (already in results CSV).
    """
    run_name = f"{dataset}_tcrp_T{T}_H{H}_{strategy}"

    # Skip if already recorded
    if skip_existing and RESULTS_CSV.exists():
        df_prev = pd.read_csv(RESULTS_CSV)
        if run_name in df_prev["run_name"].values:
            print(f"  [skip] {run_name} already in {RESULTS_CSV}")
            return None

    print(f"\n{'='*60}")
    print(f"  {run_name}")
    print(f"{'='*60}")

    cfg = build_cfg(dataset, strategy)
    results = train_run(cfg)

    row = {
        "run_name":        run_name,
        "dataset":         dataset,
        "strategy":        strategy,
        "T":               T,
        "H":               H,
        "lambda4":         STRATEGIES[strategy]["lambda4"],
        "adversarial":     STRATEGIES[strategy]["adversarial"],
        "test_mse":        results.get("test_mse",        float("nan")),
        "test_mae":        results.get("test_mae",        float("nan")),
        "test_rmse":       results.get("test_rmse",       float("nan")),
        "val_mse":         results.get("val_mse",         float("nan")),
        "elapsed_s":       results.get("total_elapsed_s", float("nan")),
        "checkpoint_path": results.get("checkpoint_path", ""),
    }

    # Append to CSV
    df_row = pd.DataFrame([row])
    if RESULTS_CSV.exists():
        df_row.to_csv(RESULTS_CSV, mode="a", header=False, index=False)
    else:
        df_row.to_csv(RESULTS_CSV, index=False)

    print(f"  test_mse={row['test_mse']:.6f}  test_mae={row['test_mae']:.6f}  rmse={row['test_rmse']:.6f}")
    print(f"  ckpt  {row['checkpoint_path']}")
    return row

## 3. Run: Standard (lambda4=0)

In [ ]:
print(DATASETS)
for ds in DATASETS:
    run_experiment(ds, "standard")

['ETTh1', 'ETTm2', 'Weather', 'ExchangeRate']

  ETTh1_tcrp_T336_H96_standard
  Training [tcrp] — ETTh1_tcrp_T336_H96_standard
  device=cuda  seed=42  lr=0.001  clip=1.0
Splits  train:10,021  val:3,053  test:3,053  batches/epoch:314
Params  91,412  K=20  N~64 segments/window

 Epoch |  train_MSE |    val_MSE |      align |         lr
--------------------------------------------------------------
     1 |   0.287007 |  12.623799 |   3.849493 |   1.00e-03
     2 |   0.201264 |  18.534661 |   2.851615 |   1.00e-03
     3 |   0.195702 |  14.003495 |   2.618930 |   1.00e-03
     4 |   0.188541 |  14.038928 |   1.931638 |   1.00e-03
     5 |   0.181513 |  23.879710 |   1.767168 |   1.00e-03
     6 |   0.162220 |  15.919566 |   1.334333 |   1.00e-03
     7 |   0.151815 |  28.605234 |   1.065219 |   5.00e-04
     8 |   0.140046 |  32.455507 |   1.034021 |   5.00e-04
     9 |   0.137784 |  22.400232 |   0.877252 |   5.00e-04
    10 |   0.135190 |  32.891036 |   0.997377 |   5.00e-04
    11 |   

## 4. Run: Regularized (lambda4=0.1)

In [ ]:
for ds in DATASETS:
    run_experiment(ds, "regularized")

## 5. Run: Adversarial (lambda4=0.1, adversarial=true)

In [ ]:
for ds in DATASETS:
    run_experiment(ds, "adversarial")

## 6. Results

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv(RESULTS_CSV)
df = df.drop_duplicates(subset="run_name", keep="last")

pivot_mse  = df.pivot(index="dataset", columns="strategy", values="test_mse")
pivot_mae  = df.pivot(index="dataset", columns="strategy", values="test_mae")
pivot_rmse = df.pivot(index="dataset", columns="strategy", values="test_rmse")

for label, pivot in [("Test MSE", pivot_mse), ("Test MAE", pivot_mae), ("Test RMSE", pivot_rmse)]:
    print(f"=== {label} ===")
    display(pivot.round(6))
    print()

In [ ]:
# ── Grouped bar chart: MSE per dataset, one bar per strategy ─────────────────
strategies = ["standard", "regularized", "adversarial"]
colors = {"standard": "#4878cf", "regularized": "#f28e2b", "adversarial": "#59a14f"}

datasets_present = [d for d in df["dataset"].unique() if d in pivot_mse.index]
x = np.arange(len(datasets_present))
w = 0.25

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (metric, pivot) in zip(axes, [("Test MSE", pivot_mse), ("Test MAE", pivot_mae)]):
    for i, strat in enumerate(strategies):
        if strat not in pivot.columns:
            continue
        vals = [pivot.loc[d, strat] if d in pivot.index else float("nan") for d in datasets_present]
        ax.bar(x + (i - 1) * w, vals, width=w, label=strat, color=colors[strat], alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(datasets_present, rotation=20, ha="right", fontsize=9)
    ax.set_title(metric, fontsize=11)
    ax.set_ylabel(metric)
    ax.legend(fontsize=8)

fig.suptitle(f"TCRP T={T} H={H} — Strategy Comparison", fontsize=12)
plt.tight_layout()
out = Path("results/strategy_comparison.png")
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out}")

In [ ]:
# ── Relative improvement over standard baseline ───────────────────────────────
if "standard" in pivot_mse.columns:
    rel = pivot_mse.copy()
    for col in ["regularized", "adversarial"]:
        if col in rel.columns:
            rel[col] = (pivot_mse["standard"] - pivot_mse[col]) / pivot_mse["standard"] * 100
    rel = rel.drop(columns="standard", errors="ignore")

    print("=== MSE improvement over standard (%) — positive = better ===")
    display(rel.round(2))

    fig, ax = plt.subplots(figsize=(10, 4))
    rel.plot(kind="bar", ax=ax,
             color=[colors["regularized"], colors["adversarial"]][:len(rel.columns)],
             rot=20)
    ax.axhline(0, color="black", lw=0.8, ls="--")
    ax.set_ylabel("Δ MSE vs standard (%)")
    ax.set_title("Relative MSE improvement over standard baseline")
    ax.legend(fontsize=9)
    plt.tight_layout()
    out = Path("results/relative_improvement.png")
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {out}")

In [ ]:
# ── Full results table sorted by test_mse ─────────────────────────────────────
display(
    df[["dataset", "strategy", "test_mse", "test_mae", "val_mse"]]
    .sort_values(["dataset", "test_mse"])
    .round(6)
    .reset_index(drop=True)
)